# MeTRAbs remote gesture server

Hosts the same MeTRAbs pose model `workers/gesture_worker.py` runs locally, but on Colab's GPU (a T4, typically) instead of your laptop's CPU — see `wikis/Gesture-Worker.md`'s "Remote MeTRAbs, local fallback" section for the full design.

Each request here is **one time window's worth of frames** (up to 150), not one frame and not a whole video — see that same wiki section for why. This notebook does its own subject-selection (vote-once/track-thereafter) exactly like `GestureWorker._process_window_local` does — **keep the two in sync** if that logic ever changes.

**Before running:**
1. Runtime -> Change runtime type -> pick a GPU (T4 is fine, free tier).
2. Sign up for a free ngrok account at https://dashboard.ngrok.com/signup and grab your authtoken from https://dashboard.ngrok.com/get-started/your-authtoken — paste it into the `NGROK_AUTHTOKEN` cell below.
3. Pick your own `API_KEY` below (any random string) and copy the *same* value into your local `.env` as `GESTURE_API_KEY` — this is the only thing stopping a random person who guesses your ngrok URL from using your compute.
4. Set `GESTURE_REMOTE_URL` locally to `{public_url}/process_window` (printed by the server cell below) once it's running.

**Session lifetime:** same as `sensevoice_server.ipynb` — Colab's free tier disconnects after inactivity and has a hard runtime cap (~12h). Not a permanently-on service; re-run this notebook whenever you want to use it again, and update `GESTURE_REMOTE_URL` locally to match the new URL. If it's ever unreachable, `GestureWorker` falls back to running MeTRAbs locally automatically — no separate action needed on a dead tunnel, just slower for that video.

In [ ]:
!pip install -q tensorflow fastapi "uvicorn[standard]" pyngrok

In [ ]:
# --- Configuration — edit these two before running ---
NGROK_AUTHTOKEN = "PASTE_YOUR_NGROK_AUTHTOKEN_HERE"
API_KEY = "PASTE_A_RANDOM_SECRET_HERE"  # must match GESTURE_API_KEY in your local .env

assert NGROK_AUTHTOKEN != "PASTE_YOUR_NGROK_AUTHTOKEN_HERE", "Set NGROK_AUTHTOKEN above first"
assert API_KEY != "PASTE_A_RANDOM_SECRET_HERE", "Set API_KEY above first — pick any random string"

In [ ]:
import tensorflow as tf

print("GPU available:", tf.config.list_physical_devices('GPU'))

# Same checkpoint as the local fallback (workers/gesture_worker.py) — keep
# these in sync so remote and local produce consistent output. ~50MB, so
# no need for the Drive-caching trick sensevoice_server.ipynb uses for its
# much larger (~900MB) model.
!curl -sL -o /tmp/metrabs.zip https://omnomnom.vision.rwth-aachen.de/data/metrabs/metrabs_mob3s_y4t.zip
!mkdir -p /content/models && unzip -q /tmp/metrabs.zip -d /content/models

model = tf.saved_model.load('/content/models/metrabs_mob3s_y4t')
print("MeTRAbs loaded and warm.")

In [ ]:
import base64
import math
import traceback

import cv2
import numpy as np
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel

# Must match workers/gesture_worker.py exactly.
_SKELETON = "coco_19"
_NUM_LANDMARKS = 19
_FRAME_CENTER = (0.5, 0.5)
_DEFAULT_FOV_DEGREES = 55.0
# If the nearest-to-ref_pos candidate is farther than this (normalised
# [0,1] frame-fraction distance), treat it as an implausible jump -- track
# loss, not a real continuation -- and re-vote by centrality instead.
_MAX_TRACK_JUMP = 0.3


def _dist(a, b):
    return math.sqrt((a[0] - b[0]) ** 2 + (a[1] - b[1]) ** 2)


def _box_center(box, width, height):
    """Detection box is [x, y, w, h, confidence] in pixel coords."""
    x, y, w, h = box[0], box[1], box[2], box[3]
    return ((x + w / 2.0) / width, (y + h / 2.0) / height)


class FrameIn(BaseModel):
    frame_idx: int
    ts: float
    jpeg_b64: str


class ProcessWindowRequest(BaseModel):
    width: int
    height: int
    frames: list[FrameIn]
    # Scene-cut timestamps (seconds) inside this window, computed client-side
    # by workers/gesture_worker.py's own independent PySceneDetect pass --
    # the server can't compute this itself, it only ever receives one
    # window's worth of already-extracted frames, never the source video
    # file, and PySceneDetect needs sequential whole-video access to work
    # at all. Defaults to [] so older clients that don't send it still work.
    cuts: list[float] = []


app = FastAPI()


@app.get("/health")
def health():
    return {"status": "ok", "gpu": len(tf.config.list_physical_devices('GPU')) > 0}


@app.post("/process_window")
def process_window(req: ProcessWindowRequest, x_api_key: str = Header(None)):
    if x_api_key != API_KEY:
        raise HTTPException(401, "bad or missing X-API-Key header")
    if not req.frames:
        return {"frames": []}

    try:
        # One HTTP request per window (unchanged -- keeps round-trips to
        # dozens per video, not hundreds/thousands), but the *model* is
        # called per frame here, not batched via detect_poses_batched.
        #
        # Batching was tried first and measured to cause real, severe
        # memory growth: detect_poses_batched is a tf.function internally,
        # so every *new* batch shape it sees gets a freshly-compiled graph
        # permanently cached for the life of the process -- confirmed
        # directly (RSS grew from 1.87GB to 7.86GB, monotonically, over
        # just 12 calls with realistic varying window frame counts, since a
        # window's frame count varies almost every request). A fixed-size
        # padding fix was tried next and *did* plateau that specific
        # growth locally, but didn't resolve the real Colab-side crash --
        # apparently something else was still accumulating there.
        #
        # Per-frame detect_poses calls have a *constant* input shape
        # ([H, W, 3], fixed for a whole video's resolution) -- the same
        # design workers/gesture_worker.py's _process_window_local already
        # uses locally, which has never shown this problem. This is slower
        # per window than one batched call would have been (less GPU
        # utilization from batching), but that's a real trade worth making
        # given the alternative was fast until it crashed.
        ref_pos = None
        next_cut_idx = 0
        out_frames = []
        for f in req.frames:
            # A cut landing anywhere at-or-before this frame's timestamp
            # invalidates whatever was being tracked -- same reasoning as
            # workers/gesture_worker.py's _process_window_local.
            while next_cut_idx < len(req.cuts) and req.cuts[next_cut_idx] <= f.ts:
                ref_pos = None
                next_cut_idx += 1

            jpeg_bytes = base64.b64decode(f.jpeg_b64)
            arr = np.frombuffer(jpeg_bytes, dtype=np.uint8)
            bgr = cv2.imdecode(arr, cv2.IMREAD_COLOR)
            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
            image = tf.constant(rgb, dtype=tf.uint8)

            pred = model.detect_poses(
                image, skeleton=_SKELETON, default_fov_degrees=_DEFAULT_FOV_DEGREES
            )
            boxes = pred["boxes"].numpy()

            if len(boxes) == 0:
                out_frames.append({
                    "frame_idx": f.frame_idx, "ts": f.ts,
                    "pose_2d": None, "pose_3d": None, "visibility": None,
                })
                continue

            poses2d = pred["poses2d"].numpy()
            poses3d = pred["poses3d"].numpy()

            # Vote-once/track-thereafter selection -- duplicated from
            # workers/gesture_worker.py's _process_window_local. Statelessly
            # safe per window: the vote always resets fresh at a window's
            # start (and at any cut within it, above), never carried across
            # windows, so no state needs to persist across requests.
            centers = [_box_center(b, req.width, req.height) for b in boxes]
            if ref_pos is None:
                chosen = min(range(len(centers)), key=lambda j: _dist(centers[j], _FRAME_CENTER))
            else:
                chosen = min(range(len(centers)), key=lambda j: _dist(centers[j], ref_pos))
                if _dist(centers[chosen], ref_pos) > _MAX_TRACK_JUMP:
                    # Implausible jump -- more likely a track switch onto
                    # someone/something else than real motion. Treat as
                    # track loss: re-vote by centrality instead of trusting it.
                    chosen = min(range(len(centers)), key=lambda j: _dist(centers[j], _FRAME_CENTER))
            ref_pos = centers[chosen]

            xy2d, xyz3d = poses2d[chosen], poses3d[chosen]
            pose_2d, pose_3d, visibility = [], [], []
            for k in range(_NUM_LANDMARKS):
                x2, y2 = float(xy2d[k][0]), float(xy2d[k][1])
                vis = 1.0 if (0.0 <= x2 < req.width and 0.0 <= y2 < req.height) else 0.0
                pose_2d.append([x2, y2])
                pose_3d.append([float(xyz3d[k][0]), float(xyz3d[k][1]), float(xyz3d[k][2])])
                visibility.append(vis)

            out_frames.append({
                "frame_idx": f.frame_idx, "ts": f.ts,
                "pose_2d": pose_2d, "pose_3d": pose_3d, "visibility": visibility,
            })

        return {"frames": out_frames}
    except Exception:
        # Surface the real traceback in the response instead of a generic
        # 500 -- there's no other way to see what actually failed here from
        # outside this Colab session.
        tb = traceback.format_exc()
        print(tb)  # also visible in this notebook's own cell output
        raise HTTPException(500, detail=tb)


In [ ]:
import threading
import time

import uvicorn
from pyngrok import conf, ngrok

conf.get_default().auth_token = NGROK_AUTHTOKEN
public_url = ngrok.connect(8000, "http")
print(f"\n  Public URL: {public_url}\n")
print(f"  Set locally: GESTURE_REMOTE_URL={public_url}/process_window\n")

# Background thread, not uvicorn.run() directly — same reasoning as
# sensevoice_server.ipynb: Colab's notebook kernel already runs its own
# asyncio event loop, and uvicorn.run()'s internal asyncio.run() call
# conflicts with that. A new thread has no event loop of its own, so this
# sidesteps the conflict instead of patching around it (nest_asyncio
# doesn't reliably work with newer uvicorn/Python 3.11+ internals).
config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)
thread = threading.Thread(target=server.run, daemon=True)
thread.start()
time.sleep(2)

print("Server running in the background thread. Run the next cell to confirm it's up.")

In [ ]:
# Sanity check — confirms the server + ngrok tunnel are both actually up
# before you go copy the URL into your local .env.
import requests

resp = requests.get(f"{public_url.public_url}/health", timeout=10)
resp.raise_for_status()
print(resp.json())